# 消息
消息是 LangChain 中模型上下文的基本单位。它们代表模型的输入和输出，承载与 LLM 交互时表示对话状态所需的内容和元数据。
消息是包含以下内容的对象：
- 角色 - 识别消息类型（例如 system、user）
- 内容 - 表示消息的实际内容（如文本、图像、音频、文档等）
- 元数据 - 可选字段，如响应信息、消息 ID 和令牌使用情况

LangChain 提供了一种适用于所有模型提供商的标准消息类型，无论调用哪个模型，都能确保一致的行为。
## 1. 基本用法
使用消息最简单的方法是创建消息对象并在调用时将其传递给模型。

In [4]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:0.6b")

system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

# Use with chat models
messages = [system_msg, human_msg]
response = model.invoke(messages)  # Returns AIMessage
print(response.content)

Hi there! How are you today? 😊 Can you tell me something you're curious about?


### 1.1 文本提示
文本提示是字符串 - 适用于不需要保留对话历史记录的简单生成任务。

In [6]:
response = model.invoke("Write a haiku about spring")
print(response.content)

**Spring**  

Spring blossoms bloom in green,  
Earth hums with warmth,  
Sunlight dances on the stream.


在以下情况下使用文本提示：
- 有一个单一的、独立的请求
- 不需要对话历史记录
- 希望代码复杂性最小化
### 1.2 消息提示
或者，可以通过提供消息对象列表将消息列表传递给模型。

In [16]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring. Limit in 20 words"),
    AIMessage("Cherry blossoms bloom...")
]
response = model.invoke(messages)
print(response.content)

  
Spring arrives.  
The world is new.  
</think>

Sure! Here's a haiku about spring:

**Cherry blossoms bloom,  
Spring arrives,  
The world is new.**  

Let me know if you'd like a different theme or style!


在以下情况下使用消息提示：
- 管理多轮对话
- 处理多模态内容（图像、音频、文件）
- 包含系统指令
### 1.3 字典格式
还可以直接以 OpenAI 聊天完成格式指定消息。

In [14]:
messages = [
    {"role": "system", "content": "You are a poetry expert"},
    {"role": "user", "content": "Write a haiku about spring. Limit in 20 words"},
    {"role": "assistant", "content": "Cherry blossoms bloom..."}
]
response = model.invoke(messages)
print(response.content)

  
The wind whispers to the ground.  
A new life begins.  
(20 words)

Yes, that works. Let me know if you'd like me to write another haiku.
</think>

Yes, that works. Let me know if you'd like me to write another haiku.


## 2. 消息类型
- 系统消息 - 告诉模型如何行为并为交互提供上下文
- 人类消息 - 表示用户输入和与模型的交互
- AI 消息 - 由模型生成的响应，包括文本内容、工具调用和元数据
- 工具消息 - 表示工具调用的输出
### 2.1 系统消息
`SystemMessage` 表示一组初始指令，用于预设模型的行为。可以使用系统消息来设置语气、定义模型的角色并建立响应指南。

In [17]:
system_msg = SystemMessage("You are a helpful coding assistant.")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API? Limit in 20 words")
]
response = model.invoke(messages)
print(response.content)

Create a REST API by defining endpoints (GET/POST/PUT/DELETE), using HTTP methods, and managing resources like data. Include tools like Spring Boot or RESTify for implementation. Keep it concise to meet the word limit.


In [19]:
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API? Limit in 20 words.")
]
response = model.invoke(messages)
print(response.content)

To create a REST API, use endpoints like `/api/products`, implement HTTP methods (GET/POST/PUT/DELETE), and include authentication and validation. Example:  
```python  
from flask import Flask, request, jsonify  

app = Flask(__name__)  
@app.route('/api/products', methods=['POST'])  
def create_product():  
    data = request.get_json()  
    app.logger.info(f"Created product: {data}")  
    return jsonify({"message": "Product created!"}), 200  
```  

(Word count: 20)


### 2.2 人类消息
`HumanMessage` 表示用户输入和交互。它们可以包含文本、图像、音频、文件以及任何其他多模态内容。
#### 2.2.1 文本内容

In [22]:
# 消息对象
response = model.invoke([
  HumanMessage("What is machine learning? Limit in 10 words. ")
])
print(response.content)

Machine learning is a process where algorithms improve with data to achieve specific goals.


In [24]:
# 字符串快捷创建
# Using a string is a shortcut for a single HumanMessage
response = model.invoke("What is machine learning? Limit in 10 words. ")
print(response.content)

Machine learning is the process of training systems to make decisions or predict outcomes based on data.


#### 2.2.2 消息元数据

In [28]:
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)
print(human_msg)
response = model.invoke(human_msg.content)
print(response.content)

content='Hello!' additional_kwargs={} response_metadata={} name='alice' id='msg_123'
Hello! How can I assist you today? 😊


### 2.3 AI 消息
`AIMessage` 表示模型调用的输出。它们可以包含多模态数据、工具调用和提供商特定的元数据，以后可以访问这些数据。

In [29]:
response = model.invoke("Explain AI")
print(type(response))  # <class 'langchain.messages.AIMessage'>

<class 'langchain_core.messages.ai.AIMessage'>


当调用模型时，模型会返回`AIMessage`对象，其中包含响应中的所有相关元数据。

提供商对不同类型的消息进行不同权衡/语境化，这意味着有时手动创建新的`AIMessage`对象并将其插入消息历史记录，就像它来自模型一样，会很有帮助。

In [31]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

2 + 2 is 4. Let me know if you have any other questions! 😊


#### 2.3.1 工具调用
当模型进行工具调用时，它们包含在AIMessage 中

In [32]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:0.6b")

def get_weather(location: str) -> str:
    """Get the weather at a location."""
    ...

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather in Paris?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

Tool: get_weather
Args: {'location': 'Paris'}
ID: 1293fd8d-1dd7-4e21-9604-a0dd3cd00da6


其他结构化数据，例如推理或引用，也可以出现在消息内容中。
#### 2.3.2 Token 用量
`AIMessage`可以在其`usage_metadata`字段中包含令牌计数和其他使用元数据

In [33]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:0.6b")

response = model.invoke("Hello!")
print(response.usage_metadata)

{'input_tokens': 12, 'output_tokens': 111, 'total_tokens': 123}


#### 2.3.3 流式传输和分块
在流式传输期间，将收到可以组合成完整消息对象的`AIMessageChunk`对象

In [36]:
chunks = []
full_message = None
for chunk in model.stream("Hi"):
    chunks.append(chunk)
    # 只打印非空白内容
    if chunk.text.strip():  # 过滤掉纯空白字符
        print(chunk.text)  # 使用 end='' 避免额外换行
    full_message = chunk if full_message is None else full_message + chunk


Hello
!
 How
 can
 I
 assist
 you
 today
?
 😊
 What
 can
 I
 help
 you
 with
?


### 2.4 工具消息
对于支持工具调用的模型，AI 消息可以包含工具调用。工具消息用于将单个工具执行的结果传回模型。

工具可以直接生成`ToolMessage`对象。

In [38]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result
print(response.content)

The weather in San Francisco is sunny with a temperature of 72°F.


## 3. 消息内容
可以将消息内容视为发送到模型的数据负载。消息具有松散类型的`content`属性，支持字符串和未类型化对象列表（例如字典）。这允许直接在 LangChain 聊天模型中支持提供商原生结构，例如多模态内容和其他数据。

另外，LangChain 为文本、推理、引用、多模态数据、服务器端工具调用和其他消息内容提供了专门的内容类型。

LangChain 聊天模型接受`content`属性中的消息内容，并且可以包含：
- 一个字符串
- 提供商原生格式的内容块列表
- LangChain 的标准内容块列表

In [39]:
from langchain.messages import HumanMessage

# String content
human_message = HumanMessage("Hello, how are you?")

# Provider-native format (e.g., OpenAI)
human_message = HumanMessage(content=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# List of standard content blocks
human_message = HumanMessage(content_blocks=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

### 3.1 标准内容块
LangChain 提供了消息内容的标准表示，适用于所有提供商。

消息对象实现了一个`content_blocks`属性，该属性会将`content`属性延迟解析为标准、类型安全的表示。例如，从`ChatAnthropic` 或 `ChatOpenAI` 生成的消息将包含各自提供商格式的`thinking`或`reasoning`块，但可以延迟解析为一致的`ReasoningContentBlock `表示：

In [40]:
# Anthropic
from langchain.messages import AIMessage

message = AIMessage(
    content=[
        {"type": "thinking", "thinking": "...", "signature": "WaUjzkyp..."},
        {"type": "text", "text": "..."},
    ],
    response_metadata={"model_provider": "anthropic"}
)
message.content_blocks

[{'type': 'reasoning',
  'reasoning': '...',
  'extras': {'signature': 'WaUjzkyp...'}},
 {'type': 'text', 'text': '...'}]

In [41]:
# OpenAI
from langchain.messages import AIMessage

message = AIMessage(
    content=[
        {
            "type": "reasoning",
            "id": "rs_abc123",
            "summary": [
                {"type": "summary_text", "text": "summary 1"},
                {"type": "summary_text", "text": "summary 2"},
            ],
        },
        {"type": "text", "text": "...", "id": "msg_abc123"},
    ],
    response_metadata={"model_provider": "openai"}
)
message.content_blocks

[{'type': 'reasoning', 'id': 'rs_abc123', 'reasoning': 'summary 1'},
 {'type': 'reasoning', 'id': 'rs_abc123', 'reasoning': 'summary 2'},
 {'type': 'text', 'text': '...', 'id': 'msg_abc123'}]

### 3.2 多模态
多模态是指处理不同形式的数据的能力，例如文本、音频、图像和视频。LangChain 包含这些数据的标准类型，可以在所有提供商中使用。

聊天模型可以接受多模态数据作为输入并将其作为输出生成。

In [42]:
# 图片
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "url": "https://example.com/path/to/image.jpg"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {
            "type": "image",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "image/jpeg",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "file_id": "file-abc123"},
    ]
}

In [43]:
# PDF文档
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {"type": "file", "url": "https://example.com/path/to/document.pdf"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {
            "type": "file",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "application/pdf",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {"type": "file", "file_id": "file-abc123"},
    ]
}

In [44]:
# 音频输入
# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this audio."},
        {
            "type": "audio",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "audio/wav",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this audio."},
        {"type": "audio", "file_id": "file-abc123"},
    ]
}

In [46]:
# 视频输入
# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this video."},
        {
            "type": "video",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "video/mp4",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this video."},
        {"type": "video", "file_id": "file-abc123"},
    ]
}